In [ ]:
from pyspark.sql import SparkSession
from utils.config_loader import load_config
from utils.logger import setup_logging, get_logger
from transformation.bronze_to_silver import run_silver_transformation
from datetime import date

setup_logging()
logger = get_logger(__name__)
spark = SparkSession.builder.appName("silver_cleaning").getOrCreate()
app_config = load_config()

# Run Bronze -> Silver for today's partition

In [ ]:
run_silver_transformation(
    spark=spark,
    bronze_table_path="Tables/bronze_job_postings",
    reference_path="Files/config/reference/country_codes.csv",
    silver_table_path="Tables/silver_job_postings",
    quarantine_table_path="Tables/silver_job_postings_quarantine",
    ingestion_date=str(date.today()),
)

# Inspect results

In [ ]:
silver_df = spark.read.format("delta").load("Tables/silver_job_postings")
print("Silver row count:", silver_df.count())
silver_df.groupBy("source").count().show()
silver_df.filter(F.size(F.col("dq_warnings")) > 0).select(
    "title", "company", "dq_warnings"
).show(10, truncate=60)